In [2]:
import sqlite3
import pandas as pd

# Connecting to SQLite Database

In [5]:
conn = sqlite3.connect("data/cleaned_data.sqlite")

# Joining ACS and PUMA Data Tables
Joining these tables through the serial_number column combines demographic and housing data. 

In [6]:
query = """
SELECT *
FROM acs_data AS a
JOIN puma_data AS p
ON a.serial_number = p.serial_number
"""

Loading the query into a DataFrame.

In [9]:
microdata_df = pd.read_sql_query(query, conn)

microdata_df.head()

,serial_number,occupation,age,sex,race,hispanic_origin,education_level,state_fips,puma_area,employment_status,wage_income,class_of_worker,hours_per_week,needs_support,serial_number,group_quarters_type
0,2022GQ0000029,Not employed,10,Male,White,Not Hispanic or Latino,Grade 5,Kentucky,Lake Cumberland Area Development District (West),Unknown,0.0,Not employed,0.0,1,2022GQ0000029,Institutional group quarters
1,2022GQ0000118,Not employed,89,Female,White,Not Hispanic or Latino,Grade 9,Kentucky,Kentucky River Area Development District,Under 16 years old,0.0,Not employed,0.0,1,2022GQ0000118,Institutional group quarters
2,2022GQ0000184,Other mathematical science occupations,22,Male,White,Not Hispanic or Latino,High school graduate or GED,Kentucky,Bluegrass Area Development District (South),Under 16 years old,10000.0,Not employed,35.0,1,2022GQ0000184,Noninstitutional group quarters
3,2022GQ0000384,Cooks,48,Male,White,Not Hispanic or Latino,High school graduate or GED,Kentucky,KIPDA Area Development District (West)--Louisv...,Under 16 years old,0.0,Not employed,0.0,1,2022GQ0000384,Noninstitutional group quarters
4,2022GQ0000396,Not employed,75,Female,White,Not Hispanic or Latino,Associate's degree,Kentucky,KIPDA Area Development District (West)--Louisv...,Under 16 years old,0.0,Not employed,0.0,1,2022GQ0000396,Institutional group quarters


## Labeling Incarcerated Individuals

**Issue:**
Housing data contains the value "Institutional group quarters", which could include prisons and jails but also medical facilities.  

**Plan:**
Now that the demographic and housing tables are merged, I can compare housing type to other demographic information to determine high likelihood that an individual is incarcerated.  
This information will be stored in a new boolean column.

### Criteria for `likely_incarcerated = True`

The `likely_incarcerated` flag is intended to identify individuals who are likely incarcerated, based on the following criteria:

- **`group_quarters_type` is `'Institutional group quarters'`**  
  Indicates the person lives in an institutional setting, which may include prisons, nursing homes, or similar facilities.

- **`age` is between 18 and 64**  
  Most nursing home and hospice residents are over 65, so this helps filter out elderly populations.

- **`wage_income` is `0`**  
  Many incarcerated individuals have no reported wage income.

Together, these filters aim to conservatively estimate incarceration status using available variables while minimizing false positives from other institutional settings.


In [26]:
microdata_df['likely_incarcerated'] = (
    (microdata_df['group_quarters_type'] == 'Institutional group quarters') &
    (microdata_df['age'].between(18, 64)) &
    (microdata_df['wage_income'] == 0)
)

microdata_df.head()


,serial_number,occupation,age,sex,race,hispanic_origin,education_level,state_fips,puma_area,employment_status,wage_income,class_of_worker,hours_per_week,needs_support,serial_number,group_quarters_type,likely_incarcerated
0,2022GQ0000029,Not employed,10,Male,White,Not Hispanic or Latino,Grade 5,Kentucky,Lake Cumberland Area Development District (West),Unknown,0.0,Not employed,0.0,1,2022GQ0000029,Institutional group quarters,False
1,2022GQ0000118,Not employed,89,Female,White,Not Hispanic or Latino,Grade 9,Kentucky,Kentucky River Area Development District,Under 16 years old,0.0,Not employed,0.0,1,2022GQ0000118,Institutional group quarters,False
2,2022GQ0000184,Other mathematical science occupations,22,Male,White,Not Hispanic or Latino,High school graduate or GED,Kentucky,Bluegrass Area Development District (South),Under 16 years old,10000.0,Not employed,35.0,1,2022GQ0000184,Noninstitutional group quarters,False
3,2022GQ0000384,Cooks,48,Male,White,Not Hispanic or Latino,High school graduate or GED,Kentucky,KIPDA Area Development District (West)--Louisv...,Under 16 years old,0.0,Not employed,0.0,1,2022GQ0000384,Noninstitutional group quarters,False
4,2022GQ0000396,Not employed,75,Female,White,Not Hispanic or Latino,Associate's degree,Kentucky,KIPDA Area Development District (West)--Louisv...,Under 16 years old,0.0,Not employed,0.0,1,2022GQ0000396,Institutional group quarters,False


In [28]:
#Checking these filters. How many people are in institutional group quarters?
microdata_df["group_quarters_type"].value_counts()

group_quarters_type
Housing unit                       44361
Institutional group quarters        1204
Noninstitutional group quarters     1040
Name: count, dtype: int64

In [29]:
#How many people in institutional group quarters have been labeled likely incarcerated?
microdata_df["likely_incarcerated"].value_counts()

likely_incarcerated
False    46101
True       504
Name: count, dtype: int64

## Saving Merged Microdata as a Table

In [32]:
#Dropping duplicated serial_number column.
microdata_df = microdata_df.loc[:, ~microdata_df.columns.duplicated()]

#Saving data as a table in the database.
microdata_df.to_sql('acs_puma_merged', conn, if_exists='replace', index=False)

46605